# 🏦 American Express — Advanced Data Analysis
> **A comprehensive financial intelligence platform covering 2016–2025**

---

## 📌 Table of Contents
1. [Setup & Configuration](#1-setup)
2. [Financial Performance Overview](#2-financial-overview)
3. [Revenue Deep-Dive & Forecasting](#3-revenue)
4. [Quarterly Trend Analysis](#4-quarterly)
5. [Segment Analysis](#5-segments)
6. [Customer & Network Metrics](#6-customer)
7. [Competitor Benchmarking](#7-competitor)
8. [Risk & Credit Quality](#8-risk)
9. [Machine Learning: Revenue Forecasting](#9-ml)
10. [Executive Dashboard Export](#10-dashboard)

---
**Data Sources:** American Express SEC Filings, Annual Reports, MacroTrends, SEC EDGAR  
**Period:** FY2016 – FY2025 (Q1 2025 TTM included)  
**Last Updated:** May 2026


In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 1: SETUP & CONFIGURATION
# ═══════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Optional advanced libraries
try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    PLOTLY = True
    print('✅ Plotly available — interactive charts enabled')
except ImportError:
    PLOTLY = False
    print('ℹ️  Plotly not found — using Matplotlib')

try:
    from sklearn.linear_model import LinearRegression
    from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
    from sklearn.preprocessing import PolynomialFeatures
    from sklearn.model_selection import cross_val_score
    from sklearn.metrics import mean_absolute_error, r2_score
    ML = True
    print('✅ Scikit-learn available — ML forecasting enabled')
except ImportError:
    ML = False
    print('ℹ️  Scikit-learn not found — install with: pip install scikit-learn')

# ── Brand Color Palette ──────────────────────────────────────
AMEX_BLUE    = '#016FD0'   # Amex primary blue
AMEX_GOLD    = '#C6A84C'   # Amex gold
AMEX_DARK    = '#0F1B2D'   # Dark navy
AMEX_LIGHT   = '#E8F4FD'   # Light blue
AMEX_GREEN   = '#00A878'   # Success green
AMEX_RED     = '#E05C5C'   # Alert red
AMEX_SILVER  = '#A8B2C1'   # Silver

PALETTE = [AMEX_BLUE, AMEX_GOLD, AMEX_GREEN, AMEX_RED, AMEX_SILVER, '#7B5EA7', '#F4845F']

# ── Matplotlib Style ─────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': AMEX_DARK,
    'axes.facecolor': '#162032',
    'axes.edgecolor': '#1E3A5F',
    'axes.labelcolor': '#FFFFFF',
    'text.color': '#FFFFFF',
    'xtick.color': '#A8B2C1',
    'ytick.color': '#A8B2C1',
    'grid.color': '#1E3A5F',
    'grid.linestyle': '--',
    'grid.alpha': 0.4,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.spines.left': False,
    'axes.spines.bottom': False,
})

print('\n🏦 American Express Data Analysis — Environment Ready')
print('=' * 55)

In [ ]:
# ═══════════════════════════════════════════════════════════
# LOAD ALL DATASETS
# ═══════════════════════════════════════════════════════════

df_annual   = pd.read_csv('../data/amex_annual_financials.csv')
df_quarter  = pd.read_csv('../data/amex_quarterly_data.csv')
df_segments = pd.read_csv('../data/amex_segments.csv')
df_customer = pd.read_csv('../data/amex_customer_metrics.csv')
df_comp     = pd.read_csv('../data/competitor_analysis.csv')

# ── Derived Metrics ──────────────────────────────────────────
df_annual['Revenue_YoY_pct']     = df_annual['Total_Revenue_B'].pct_change() * 100
df_annual['NetIncome_YoY_pct']   = df_annual['Net_Income_B'].pct_change() * 100
df_annual['Net_Margin_pct']      = df_annual['Net_Income_B'] / df_annual['Total_Revenue_B'] * 100
df_annual['Revenue_Per_Card']    = (df_annual['Total_Revenue_B'] * 1e9) / (df_annual['Cards_In_Force_M'] * 1e6)
df_annual['Spend_Per_Card_K']    = (df_annual['Billed_Business_T'] * 1e12) / (df_annual['Cards_In_Force_M'] * 1e6) / 1000
df_annual['Card_Fee_pct_Rev']    = df_annual['Net_Card_Fee_Revenue_B'] / df_annual['Total_Revenue_B'] * 100
df_annual['P_E_Ratio']           = df_annual['Stock_Price_EOY'] / df_annual['EPS_Diluted']

print('✅ Datasets loaded successfully')
print(f'   Annual data: {df_annual.shape[0]} years ({df_annual.Year.min()}–{df_annual.Year.max()})')
print(f'   Quarterly data: {df_quarter.shape[0]} quarters')
print(f'   Segments: {df_segments.shape[0]} rows')
df_annual[['Year','Total_Revenue_B','Net_Income_B','EPS_Diluted','Net_Margin_pct','Revenue_YoY_pct']].tail(5)

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 2: KPI SCORECARD — EXECUTIVE SUMMARY
# ═══════════════════════════════════════════════════════════

latest = df_annual.iloc[-1]
prev   = df_annual.iloc[-2]

def kpi_change(curr, prev):
    chg = (curr - prev) / prev * 100
    arrow = '▲' if chg > 0 else '▼'
    color = AMEX_GREEN if chg > 0 else AMEX_RED
    return f'{arrow} {abs(chg):.1f}%', color

fig, axes = plt.subplots(2, 4, figsize=(18, 7))
fig.suptitle('AMERICAN EXPRESS — 2025 KPI SCORECARD', 
             fontsize=18, fontweight='bold', color=AMEX_GOLD, y=1.01)

kpis = [
    ('Total Revenue', f'${latest.Total_Revenue_B:.1f}B', latest.Total_Revenue_B, prev.Total_Revenue_B),
    ('Net Income',    f'${latest.Net_Income_B:.2f}B',    latest.Net_Income_B,    prev.Net_Income_B),
    ('Diluted EPS',   f'${latest.EPS_Diluted:.2f}',      latest.EPS_Diluted,     prev.EPS_Diluted),
    ('Billed Business', f'${latest.Billed_Business_T:.2f}T', latest.Billed_Business_T, prev.Billed_Business_T),
    ('Cards In Force', f'{latest.Cards_In_Force_M:.0f}M', latest.Cards_In_Force_M, prev.Cards_In_Force_M),
    ('Net Card Fees',  f'${latest.Net_Card_Fee_Revenue_B:.2f}B', latest.Net_Card_Fee_Revenue_B, prev.Net_Card_Fee_Revenue_B),
    ('Net Margin',     f'{latest.Net_Margin_pct:.1f}%',  latest.Net_Margin_pct,  prev.Net_Margin_pct),
    ('Return on Equity', f'{latest.Return_on_Equity_pct:.1f}%', latest.Return_on_Equity_pct, prev.Return_on_Equity_pct),
]

for ax, (title, value, curr, prev_val) in zip(axes.flat, kpis):
    chg_str, chg_color = kpi_change(curr, prev_val)
    ax.set_facecolor('#162032')
    ax.text(0.5, 0.72, value,    ha='center', va='center', fontsize=22, fontweight='bold', 
            color=AMEX_GOLD,  transform=ax.transAxes)
    ax.text(0.5, 0.42, chg_str,  ha='center', va='center', fontsize=13, fontweight='bold', 
            color=chg_color,  transform=ax.transAxes)
    ax.text(0.5, 0.16, title,    ha='center', va='center', fontsize=10, 
            color=AMEX_SILVER, transform=ax.transAxes)
    ax.text(0.5, 0.03, 'vs 2024', ha='center', va='center', fontsize=7.5, 
            color='#5A7A9A',   transform=ax.transAxes)
    for spine in ax.spines.values():
        spine.set_edgecolor(AMEX_BLUE)
        spine.set_linewidth(1.5)
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.savefig('../assets/kpi_scorecard.png', dpi=150, bbox_inches='tight', facecolor=AMEX_DARK)
plt.show()
print('✅ KPI Scorecard saved')

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 3: REVENUE & INCOME DEEP DIVE
# ═══════════════════════════════════════════════════════════

fig = plt.figure(figsize=(18, 10))
gs  = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

ax1 = fig.add_subplot(gs[0, :2])
ax2 = fig.add_subplot(gs[0, 2])
ax3 = fig.add_subplot(gs[1, 0])
ax4 = fig.add_subplot(gs[1, 1])
ax5 = fig.add_subplot(gs[1, 2])

years = df_annual['Year']

# ── Chart 1: Revenue + Net Income dual-axis bar ──────────────
bars = ax1.bar(years, df_annual['Total_Revenue_B'], color=AMEX_BLUE, alpha=0.85, label='Total Revenue', zorder=3)
ax1b = ax1.twinx()
ax1b.plot(years, df_annual['Net_Income_B'], color=AMEX_GOLD, marker='o', lw=2.5, ms=7, label='Net Income', zorder=4)
ax1b.fill_between(years, df_annual['Net_Income_B'], alpha=0.15, color=AMEX_GOLD)
ax1.set_title('Revenue vs Net Income (2016–2025)', fontsize=13, fontweight='bold', pad=12, color='white')
ax1.set_ylabel('Revenue ($B)', color=AMEX_BLUE)
ax1b.set_ylabel('Net Income ($B)', color=AMEX_GOLD)
ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0fB'))
ax1b.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0fB'))
ax1.legend(loc='upper left'); ax1b.legend(loc='upper right')
ax1.set_facecolor('#162032'); ax1b.set_facecolor('#162032')
ax1.grid(True, alpha=0.3)

# Add YoY % labels on bars
for bar, yoy in zip(bars, df_annual['Revenue_YoY_pct']):
    if not np.isnan(yoy):
        color = AMEX_GREEN if yoy > 0 else AMEX_RED
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'+{yoy:.0f}%' if yoy > 0 else f'{yoy:.0f}%',
                 ha='center', fontsize=7.5, color=color, fontweight='bold')

# ── Chart 2: Net Margin trend ────────────────────────────────
ax2.fill_between(years, df_annual['Net_Margin_pct'], alpha=0.3, color=AMEX_GREEN)
ax2.plot(years, df_annual['Net_Margin_pct'], color=AMEX_GREEN, lw=2.5, marker='D', ms=6)
ax2.set_title('Net Profit Margin %', fontsize=11, fontweight='bold', color='white')
ax2.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
ax2.set_facecolor('#162032'); ax2.grid(True, alpha=0.3)
for x, y in zip(years, df_annual['Net_Margin_pct']):
    ax2.annotate(f'{y:.1f}%', (x, y), textcoords='offset points', xytext=(0, 8),
                 ha='center', fontsize=7, color=AMEX_GREEN)

# ── Chart 3: EPS Growth ──────────────────────────────────────
ax3.bar(years, df_annual['EPS_Diluted'], color=AMEX_GOLD, alpha=0.85)
ax3.set_title('Diluted EPS ($)', fontsize=11, fontweight='bold', color='white')
ax3.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.2f'))
ax3.set_facecolor('#162032'); ax3.grid(True, alpha=0.3)

# ── Chart 4: Card Fee Revenue Growth ────────────────────────
ax4.plot(years, df_annual['Net_Card_Fee_Revenue_B'], color='#FF9F43', lw=2.5, marker='s', ms=6)
ax4.fill_between(years, df_annual['Net_Card_Fee_Revenue_B'], alpha=0.2, color='#FF9F43')
ax4.set_title('Net Card Fee Revenue ($B)', fontsize=11, fontweight='bold', color='white')
ax4.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.1fB'))
ax4.set_facecolor('#162032'); ax4.grid(True, alpha=0.3)

# ── Chart 5: Return on Equity ───────────────────────────────
colors_roe = [AMEX_GREEN if r > 25 else AMEX_BLUE for r in df_annual['Return_on_Equity_pct']]
ax5.bar(years, df_annual['Return_on_Equity_pct'], color=colors_roe, alpha=0.85)
ax5.axhline(y=25, color=AMEX_GOLD, lw=1.5, ls='--', label='25% target')
ax5.set_title('Return on Equity (%)', fontsize=11, fontweight='bold', color='white')
ax5.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax5.set_facecolor('#162032'); ax5.grid(True, alpha=0.3)
ax5.legend(fontsize=8)

fig.suptitle('AMERICAN EXPRESS — FINANCIAL DEEP DIVE', fontsize=16, fontweight='bold', 
             color=AMEX_GOLD, y=1.01)
fig.patch.set_facecolor(AMEX_DARK)
plt.savefig('../assets/financial_deep_dive.png', dpi=150, bbox_inches='tight', facecolor=AMEX_DARK)
plt.show()
print('✅ Financial Deep Dive chart saved')

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 4: QUARTERLY TREND ANALYSIS
# ═══════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('QUARTERLY PERFORMANCE ANALYSIS (2022–2025)', 
             fontsize=16, fontweight='bold', color=AMEX_GOLD)
fig.patch.set_facecolor(AMEX_DARK)

periods = df_quarter['Period']
x = range(len(periods))

# Color per year
year_colors = {2022: AMEX_SILVER, 2023: AMEX_BLUE, 2024: AMEX_GOLD, 2025: AMEX_GREEN}
bar_colors  = [year_colors[y] for y in df_quarter['Year']]

# Chart 1 — Quarterly Revenue
ax = axes[0,0]
bars = ax.bar(x, df_quarter['Total_Revenue_B'], color=bar_colors, alpha=0.9, width=0.7)
ax.plot(x, df_quarter['Total_Revenue_B'].rolling(4).mean(), 
        color='white', lw=2, ls='--', label='4Q Moving Avg')
ax.set_title('Quarterly Revenue ($B)', fontsize=12, color='white', fontweight='bold')
ax.set_xticks(x[::2]); ax.set_xticklabels(periods[::2], rotation=45, fontsize=7)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0fB'))
ax.set_facecolor('#162032'); ax.grid(axis='y', alpha=0.3)
ax.legend(fontsize=8)
# Year legend
patches = [mpatches.Patch(color=c, label=str(y)) for y, c in year_colors.items()]
ax.legend(handles=patches + [plt.Line2D([0],[0], color='white', ls='--', label='4Q MA')],
         fontsize=8, loc='upper left')

# Chart 2 — Quarterly EPS
ax = axes[0,1]
ax.plot(x, df_quarter['EPS_Diluted'], color=AMEX_GOLD, marker='o', lw=2.5, ms=6)
ax.fill_between(x, df_quarter['EPS_Diluted'], alpha=0.2, color=AMEX_GOLD)
ax.set_title('Quarterly EPS (Diluted)', fontsize=12, color='white', fontweight='bold')
ax.set_xticks(x[::2]); ax.set_xticklabels(periods[::2], rotation=45, fontsize=7)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.2f'))
ax.set_facecolor('#162032'); ax.grid(axis='y', alpha=0.3)

# Chart 3 — Billed Business
ax = axes[1,0]
ax.bar(x, df_quarter['Billed_Business_B'], color=bar_colors, alpha=0.85, width=0.7)
ax.set_title('Card Member Billed Business ($B)', fontsize=12, color='white', fontweight='bold')
ax.set_xticks(x[::2]); ax.set_xticklabels(periods[::2], rotation=45, fontsize=7)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0fB'))
ax.set_facecolor('#162032'); ax.grid(axis='y', alpha=0.3)

# Chart 4 — Write-off Rate
ax = axes[1,1]
write_colors = [AMEX_GREEN if r < 2 else AMEX_RED for r in df_quarter['Write_Off_Rate_pct']]
ax.bar(x, df_quarter['Write_Off_Rate_pct'], color=write_colors, alpha=0.85, width=0.7)
ax.axhline(y=2.0, color='white', lw=1.5, ls='--', label='2% threshold')
ax.set_title('Net Write-Off Rate (%)', fontsize=12, color='white', fontweight='bold')
ax.set_xticks(x[::2]); ax.set_xticklabels(periods[::2], rotation=45, fontsize=7)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
ax.set_facecolor('#162032'); ax.grid(axis='y', alpha=0.3)
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../assets/quarterly_analysis.png', dpi=150, bbox_inches='tight', facecolor=AMEX_DARK)
plt.show()
print('✅ Quarterly analysis chart saved')

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 5: SEGMENT ANALYSIS
# ═══════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.suptitle('BUSINESS SEGMENT DEEP DIVE', fontsize=16, fontweight='bold', color=AMEX_GOLD)
fig.patch.set_facecolor(AMEX_DARK)

segments = ['US Consumer Services', 'Commercial Services', 'International Card Services', 'GMNS & Other']
seg_colors = [AMEX_BLUE, AMEX_GOLD, AMEX_GREEN, AMEX_SILVER]

# Chart 1 — Revenue by segment stacked bar
ax = axes[0]
years_seg = [2022, 2023, 2024, 2025]
bottom = np.zeros(4)
for seg, color in zip(segments, seg_colors):
    vals = df_segments[df_segments['Segment'] == seg]['Revenue_B'].values
    ax.bar(years_seg, vals, bottom=bottom, label=seg, color=color, alpha=0.88, width=0.6)
    bottom += vals
ax.set_title('Revenue by Segment ($B)', fontsize=12, color='white', fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0fB'))
ax.set_facecolor('#162032'); ax.grid(axis='y', alpha=0.3)
ax.legend(fontsize=7.5, loc='upper left')

# Chart 2 — 2025 segment revenue pie
ax = axes[1]
vals_2025 = [df_segments[(df_segments['Segment']==s) & (df_segments['Year']==2025)]['Revenue_B'].values[0] 
             for s in segments]
wedges, texts, autotexts = ax.pie(
    vals_2025, labels=None, colors=seg_colors, 
    autopct='%1.1f%%', startangle=140,
    wedgeprops=dict(edgecolor=AMEX_DARK, linewidth=2),
    pctdistance=0.78
)
for at in autotexts:
    at.set_color('white'); at.set_fontsize(10); at.set_fontweight('bold')
ax.set_title('2025 Revenue Mix', fontsize=12, color='white', fontweight='bold')
ax.legend(segments, loc='lower center', bbox_to_anchor=(0.5, -0.15), fontsize=7.5, ncol=2)
ax.set_facecolor('#162032')

# Chart 3 — Pretax Income margin per segment 2025
ax = axes[2]
df_25 = df_segments[df_segments['Year']==2025]
df_25 = df_25[df_25['Segment'] != 'GMNS & Other'].copy()
df_25['Margin'] = df_25['Pretax_Income_B'] / df_25['Revenue_B'] * 100
y_pos = range(len(df_25))
bars = ax.barh(y_pos, df_25['Margin'], color=[AMEX_BLUE, AMEX_GOLD, AMEX_GREEN], alpha=0.88, height=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(df_25['Segment'], fontsize=9)
ax.set_title('2025 Pretax Margin by Segment', fontsize=12, color='white', fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.set_facecolor('#162032'); ax.grid(axis='x', alpha=0.3)
for bar, val in zip(bars, df_25['Margin']):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2, f'{val:.1f}%',
            va='center', fontsize=10, color=AMEX_GOLD, fontweight='bold')

plt.tight_layout()
plt.savefig('../assets/segment_analysis.png', dpi=150, bbox_inches='tight', facecolor=AMEX_DARK)
plt.show()
print('✅ Segment analysis chart saved')

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 6: CUSTOMER & NETWORK METRICS
# ═══════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('CUSTOMER INTELLIGENCE & NETWORK METRICS', fontsize=16, fontweight='bold', color=AMEX_GOLD)
fig.patch.set_facecolor(AMEX_DARK)

years_c = [2022, 2023, 2024, 2025]

def get_metric(df, cat, subcat):
    row = df[(df['Category'] == cat) & (df['Subcategory'] == subcat)]
    return [row['Value_2022'].values[0], row['Value_2023'].values[0], 
            row['Value_2024'].values[0], row['Value_2025'].values[0]]

# Chart 1 — Cards in force
ax = axes[0,0]
ax.plot(years_c, df_annual[df_annual['Year'].isin(years_c)]['Cards_In_Force_M'], 
        color=AMEX_BLUE, lw=3, marker='o', ms=8)
ax.fill_between(years_c, df_annual[df_annual['Year'].isin(years_c)]['Cards_In_Force_M'], 
                alpha=0.2, color=AMEX_BLUE)
ax.set_title('Cards In Force (Millions)', fontsize=11, color='white', fontweight='bold')
ax.set_facecolor('#162032'); ax.grid(alpha=0.3)
for x, y in zip(years_c, df_annual[df_annual['Year'].isin(years_c)]['Cards_In_Force_M']):
    ax.annotate(f'{y:.0f}M', (x, y), xytext=(0, 10), textcoords='offset points',
                ha='center', fontsize=9, color=AMEX_BLUE, fontweight='bold')

# Chart 2 — Merchant network growth
ax = axes[0,1]
merchant_locs = get_metric(df_customer, 'Network', 'Merchant_Locations_M')
ax.bar(years_c, merchant_locs, color=AMEX_GOLD, alpha=0.85, width=0.5)
ax.set_title('Merchant Locations (Millions)', fontsize=11, color='white', fontweight='bold')
ax.set_facecolor('#162032'); ax.grid(axis='y', alpha=0.3)
for x, y in zip(years_c, merchant_locs):
    ax.text(x, y + 1, f'{y:.0f}M', ha='center', fontsize=10, color=AMEX_GOLD, fontweight='bold')

# Chart 3 — Millennial/GenZ acquisition share
ax = axes[0,2]
gz_pct = get_metric(df_customer, 'Demographics', 'Millennial_GenZ_New_Acq_pct')
ax.bar(years_c, gz_pct, color=AMEX_GREEN, alpha=0.85, width=0.5)
ax.axhline(50, color='white', ls='--', lw=1.5, alpha=0.7)
ax.set_title('Millennial & Gen Z New Acquisitions (%)', fontsize=11, color='white', fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.set_facecolor('#162032'); ax.grid(axis='y', alpha=0.3)
for x, y in zip(years_c, gz_pct):
    ax.text(x, y + 0.5, f'{y}%', ha='center', fontsize=11, color=AMEX_GREEN, fontweight='bold')

# Chart 4 — Spending category breakdown 2025
ax = axes[1,0]
spend_cats = ['Travel & Entertainment','Dining & Food','Retail & Shopping',
              'Digital & Subscription','Healthcare','Business & B2B']
spend_vals_25 = []
for cat in spend_cats:
    row = df_customer[(df_customer['Category']=='Spending Category') & 
                     (df_customer['Subcategory']==cat)]
    if len(row) > 0:
        spend_vals_25.append(row['Value_2025'].values[0])

colors_spend = [AMEX_BLUE, AMEX_GOLD, AMEX_GREEN, '#FF9F43', AMEX_RED, AMEX_SILVER]
wedges, _, autotexts = ax.pie(spend_vals_25, labels=None, colors=colors_spend,
                               autopct='%1.1f%%', startangle=90,
                               wedgeprops=dict(edgecolor=AMEX_DARK, linewidth=2),
                               pctdistance=0.75)
for at in autotexts:
    at.set_color('white'); at.set_fontsize(9)
ax.set_title('2025 Spending by Category', fontsize=11, color='white', fontweight='bold')
ax.legend(spend_cats, loc='lower center', bbox_to_anchor=(0.5,-0.22), fontsize=7, ncol=2)
ax.set_facecolor('#162032')

# Chart 5 — Card retention rate
ax = axes[1,1]
retention = get_metric(df_customer, 'Retention', 'Card_Member_Retention_Rate_pct')
ax.fill_between(years_c, retention, 80, alpha=0.3, color=AMEX_GREEN)
ax.plot(years_c, retention, color=AMEX_GREEN, lw=3, marker='D', ms=9)
ax.set_ylim(80, 95)
ax.set_title('Card Member Retention Rate (%)', fontsize=11, color='white', fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.set_facecolor('#162032'); ax.grid(alpha=0.3)
for x, y in zip(years_c, retention):
    ax.annotate(f'{y}%', (x, y), xytext=(0, 10), textcoords='offset points',
                ha='center', fontsize=11, color=AMEX_GREEN, fontweight='bold')

# Chart 6 — Premium card share trend
ax = axes[1,2]
premium = get_metric(df_customer, 'Demographics', 'Premium_Card_Share_pct')
non_premium = [100 - p for p in premium]
ax.bar(years_c, premium, label='Premium Cards', color=AMEX_GOLD, alpha=0.88, width=0.5)
ax.bar(years_c, non_premium, bottom=premium, label='Standard Cards', color=AMEX_SILVER, alpha=0.5, width=0.5)
ax.set_title('Premium vs Standard Card Mix', fontsize=11, color='white', fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.set_facecolor('#162032'); ax.grid(axis='y', alpha=0.3)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../assets/customer_metrics.png', dpi=150, bbox_inches='tight', facecolor=AMEX_DARK)
plt.show()
print('✅ Customer metrics chart saved')

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 7: COMPETITOR BENCHMARKING
# ═══════════════════════════════════════════════════════════

df_25 = df_comp[df_comp['Year'] == 2025].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.suptitle('COMPETITIVE LANDSCAPE — 2025 BENCHMARKING', fontsize=16, fontweight='bold', color=AMEX_GOLD)
fig.patch.set_facecolor(AMEX_DARK)

companies = df_25['Company'].tolist()
comp_colors = [AMEX_GOLD if 'American' in c else AMEX_SILVER for c in companies]

# Chart 1 — Revenue comparison
ax = axes[0]
ax.barh(companies, df_25['Revenue_B'], color=comp_colors, alpha=0.88, height=0.6)
ax.set_title('Revenue ($B) — 2025', fontsize=12, color='white', fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0fB'))
ax.set_facecolor('#162032'); ax.grid(axis='x', alpha=0.3)
for i, v in enumerate(df_25['Revenue_B']):
    ax.text(v + 0.3, i, f'${v:.1f}B', va='center', fontsize=9,
            color=AMEX_GOLD if i==0 else AMEX_SILVER, fontweight='bold')

# Chart 2 — Net Income comparison
ax = axes[1]
ax.barh(companies, df_25['Net_Income_B'], color=comp_colors, alpha=0.88, height=0.6)
ax.set_title('Net Income ($B) — 2025', fontsize=12, color='white', fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0fB'))
ax.set_facecolor('#162032'); ax.grid(axis='x', alpha=0.3)
for i, v in enumerate(df_25['Net_Income_B']):
    ax.text(v + 0.1, i, f'${v:.1f}B', va='center', fontsize=9,
            color=AMEX_GOLD if i==0 else AMEX_SILVER, fontweight='bold')

# Chart 3 — Net Promoter Score radar/bar
ax = axes[2]
ax.barh(companies, df_25['Net_Promoter_Score'], color=comp_colors, alpha=0.88, height=0.6)
ax.set_title('Net Promoter Score (NPS)', fontsize=12, color='white', fontweight='bold')
ax.set_facecolor('#162032'); ax.grid(axis='x', alpha=0.3)
for i, v in enumerate(df_25['Net_Promoter_Score']):
    ax.text(v + 0.5, i, str(v), va='center', fontsize=10,
            color=AMEX_GOLD if i==0 else AMEX_SILVER, fontweight='bold')

plt.tight_layout()
plt.savefig('../assets/competitor_analysis.png', dpi=150, bbox_inches='tight', facecolor=AMEX_DARK)
plt.show()
print('✅ Competitor analysis chart saved')

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 9: MACHINE LEARNING — REVENUE FORECASTING
# ═══════════════════════════════════════════════════════════

if not ML:
    print('⚠️  Scikit-learn not available. Run: pip install scikit-learn')
else:
    from sklearn.linear_model import LinearRegression
    from sklearn.ensemble import GradientBoostingRegressor
    from sklearn.preprocessing import PolynomialFeatures
    from sklearn.metrics import r2_score, mean_absolute_error

    X = df_annual[['Year']].values
    y = df_annual['Total_Revenue_B'].values

    # Model 1: Linear
    lr = LinearRegression().fit(X, y)

    # Model 2: Polynomial degree 3
    poly = PolynomialFeatures(degree=3)
    X_poly = poly.fit_transform(X)
    pr = LinearRegression().fit(X_poly, y)

    # Forecast 2026-2028
    future_years = np.array([[2026],[2027],[2028]])
    lr_forecast  = lr.predict(future_years)
    pr_forecast  = pr.predict(poly.transform(future_years))

    all_years = np.vstack([X, future_years])
    lr_full   = lr.predict(all_years)
    pr_full   = pr.predict(poly.transform(all_years))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
    fig.suptitle('ML REVENUE FORECASTING — 2026–2028', fontsize=16, fontweight='bold', color=AMEX_GOLD)
    fig.patch.set_facecolor(AMEX_DARK)

    all_y_vals = all_years.flatten()
    split = len(X)

    ax1.scatter(df_annual['Year'], y, color=AMEX_GOLD, s=80, zorder=5, label='Actual')
    ax1.plot(all_y_vals, lr_full, color=AMEX_BLUE, lw=2, ls='--', label='Linear Model')
    ax1.plot(all_y_vals, pr_full, color=AMEX_GREEN, lw=2.5, label='Poly Degree-3')
    ax1.axvspan(2025.5, 2028.5, alpha=0.08, color=AMEX_GOLD)
    ax1.axvline(2025.5, color='white', ls=':', lw=1.5, alpha=0.6)
    ax1.text(2026, min(y)*0.9, '📈 Forecast Zone', color=AMEX_GOLD, fontsize=9, fontstyle='italic')
    ax1.set_title('Revenue Forecast Models', fontsize=12, color='white', fontweight='bold')
    ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0fB'))
    ax1.set_facecolor('#162032'); ax1.grid(alpha=0.3); ax1.legend()

    # Forecast table
    headers = ['Year', 'Linear ($B)', 'Polynomial ($B)', 'YoY Growth Est']
    rows = []
    for i, yr in enumerate([2026, 2027, 2028]):
        prev_base = pr_forecast[i-1] if i > 0 else y[-1]
        growth = (pr_forecast[i] - prev_base) / prev_base * 100
        rows.append([yr, f'${lr_forecast[i]:.1f}B', f'${pr_forecast[i]:.1f}B', f'{growth:.1f}%'])

    ax2.axis('off')
    tbl = ax2.table(cellText=rows, colLabels=headers, loc='center', cellLoc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(13)
    tbl.scale(1.4, 2.8)
    for (r, c), cell in tbl.get_celld().items():
        cell.set_facecolor('#162032' if r > 0 else '#1E3A5F')
        cell.set_edgecolor(AMEX_BLUE)
        cell.set_text_props(color=AMEX_GOLD if r==0 else 'white', fontweight='bold' if r==0 else 'normal')
    ax2.set_title('3-Year Revenue Forecast Summary', fontsize=12, color='white', fontweight='bold', pad=15)
    ax2.set_facecolor('#162032')

    r2_lr = r2_score(y, lr.predict(X))
    r2_pr = r2_score(y, pr.predict(X_poly))
    print(f'Linear R²: {r2_lr:.4f}  |  Polynomial R²: {r2_pr:.4f}')
    print(f'2026 Poly Forecast: ${pr_forecast[0]:.1f}B')
    print(f'2027 Poly Forecast: ${pr_forecast[1]:.1f}B')
    print(f'2028 Poly Forecast: ${pr_forecast[2]:.1f}B')

    plt.tight_layout()
    plt.savefig('../assets/ml_forecast.png', dpi=150, bbox_inches='tight', facecolor=AMEX_DARK)
    plt.show()
    print('✅ ML Forecast chart saved')

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 10: CORRELATION HEATMAP & FINAL SUMMARY
# ═══════════════════════════════════════════════════════════

numeric_cols = ['Total_Revenue_B','Net_Income_B','EPS_Diluted','Billed_Business_T',
                'Cards_In_Force_M','Net_Card_Fee_Revenue_B','Return_on_Equity_pct',
                'Stock_Price_EOY','Net_Margin_pct']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('CORRELATION ANALYSIS & STRATEGIC INSIGHTS', fontsize=16, fontweight='bold', color=AMEX_GOLD)
fig.patch.set_facecolor(AMEX_DARK)

corr = df_annual[numeric_cols].corr()
short_labels = ['Revenue','Net Inc','EPS','Billed Biz','Cards','Card Fees','ROE','Stock','Margin']

mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(220, 10, as_cmap=True)
sns.heatmap(corr, ax=ax1, mask=mask, cmap='RdYlGn', center=0,
            xticklabels=short_labels, yticklabels=short_labels,
            annot=True, fmt='.2f', linewidths=1, linecolor='#0F1B2D',
            cbar_kws={'shrink': 0.8})
ax1.set_title('Correlation Matrix', fontsize=12, color='white', fontweight='bold')
ax1.set_facecolor('#162032')
ax1.tick_params(colors='white', labelsize=8)

# Stock price vs Revenue scatter
ax2.scatter(df_annual['Total_Revenue_B'], df_annual['Stock_Price_EOY'],
            c=df_annual['Year'], cmap='Blues', s=120, zorder=5, edgecolors=AMEX_GOLD, lw=1.5)
for _, row in df_annual.iterrows():
    ax2.annotate(str(int(row.Year)), (row.Total_Revenue_B, row.Stock_Price_EOY),
                xytext=(4, 4), textcoords='offset points', fontsize=8, color=AMEX_SILVER)
# Trend line
z = np.polyfit(df_annual['Total_Revenue_B'], df_annual['Stock_Price_EOY'], 1)
p = np.poly1d(z)
xline = np.linspace(df_annual['Total_Revenue_B'].min(), df_annual['Total_Revenue_B'].max(), 50)
ax2.plot(xline, p(xline), color=AMEX_GOLD, lw=2, ls='--', alpha=0.7, label='Trend')
ax2.set_xlabel('Total Revenue ($B)', color='white'); ax2.set_ylabel('Stock Price (EOY, $)', color='white')
ax2.set_title('Revenue vs Stock Price', fontsize=12, color='white', fontweight='bold')
ax2.set_facecolor('#162032'); ax2.grid(alpha=0.3); ax2.legend()

plt.tight_layout()
plt.savefig('../assets/correlation_analysis.png', dpi=150, bbox_inches='tight', facecolor=AMEX_DARK)
plt.show()

print('\n' + '='*60)
print('🏦 AMERICAN EXPRESS — ANALYSIS COMPLETE')
print('='*60)
print(f"  Revenue 2025:    ${df_annual.iloc[-1]['Total_Revenue_B']:.2f}B")
print(f"  Net Income 2025: ${df_annual.iloc[-1]['Net_Income_B']:.2f}B")
print(f"  EPS 2025:        ${df_annual.iloc[-1]['EPS_Diluted']:.2f}")
print(f"  Net Margin:      {df_annual.iloc[-1]['Net_Margin_pct']:.1f}%")
print(f"  Cards In Force:  {df_annual.iloc[-1]['Cards_In_Force_M']:.0f}M")
print('='*60)
print('  ✅ All charts saved to ../assets/')
print('  📊 Dashboard ready for deployment')
print('='*60)